In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim


from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split


SEED = 42

In [10]:
# mps
# cuda
# cpu

w = torch.tensor(2.0, requires_grad=True)
y = 4 * w + 2
y.backward()
w.grad

tensor(4.)

In [11]:
x = torch.tensor([-3.0, -1.0, 0, 1.0, 3.0])

Activations Functions

In [12]:
activations = {
    "ReLU": nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(negative_slope=0.1),
    "Tanh": nn.Tanh(),
    "Swiss": nn.SiLU()
}

rows = []
for name, fn in activations.items():
    y = fn(x).numpy().round(3)
    rows.append(pd.Series(y, index=x.numpy(), name=name))

df = pd.DataFrame(rows)
df

,-3.0,-1.0,0.0,1.0,3.0
ReLU,0.000,0.000,0.0,1.000,3.000
LeakyReLU,-0.300,-0.100,0.0,1.000,3.000
Tanh,-0.995,-0.762,0.0,0.762,0.995
Swiss,-0.142,-0.269,0.0,0.731,2.858


Dropout

In [13]:
dropout = nn.Dropout(0.5)

data = torch.ones(16).reshape(4, 4)
data

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [14]:
dropout.train()  # режим навчання
dropout.eval()   # режим оцінки

data_y = dropout(data)
data_y

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [15]:
X, y = make_regression(
    n_features=20,
    n_informative=8,
    noise=20,
    random_state=SEED
)
y = y.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [17]:
model = nn.Sequential(
    # Layer 1
    nn.Linear(20, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    # Layer 2
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(0.5),
    # Output
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()


optimizer = optim.Adam(model.parameters(), lr=0.01)

In [18]:
epochs = 1000

history = []

for epoch in range(1, epochs + 1):
    model.train()

    optimizer.zero_grad()

    y_pred_tensor = model(X_train_tensor)

    loss = loss_fn(y_train_tensor, y_pred_tensor)

    loss.backward()

    optimizer.step()

    loss_train = loss.item()

    # оцінка
    model.eval()

    with torch.no_grad():
        y_test_pred_tensor = model(X_test_tensor)
        loss_test = loss_fn(y_test_tensor, y_test_pred_tensor)

    history.append({
        "epoch": epoch,
        "train_loss": loss_train,
        "test_loss": loss_test.item()
    })


In [19]:
df = pd.DataFrame(history)

In [20]:
df.tail()

,epoch,train_loss,test_loss
995,996,4629.209961,4004.759277
996,997,4125.938477,3964.567139
997,998,4873.804199,3954.069092
998,999,4911.104004,3925.897949
999,1000,4829.108398,3916.567627
